# train.py

In [ ]:
# train.py
# Обучение всех нейросетевых моделей для ATSP (MatNet, MatPOENet, PointerNetwork, AttentionModel)
# с помощью метода REINFORCE на синтетических асимметричных матрицах расстояний.
# Сохраняет чекпоинты, метрики и графики обучения.

import torch
import torch.nn as nn
import torch.optim as optim
import os
import json
import matplotlib.pyplot as plt
from tqdm import tqdm

from neural_models import (
    AttentionModelATSP, PointerNetworkATSP, MatNet, MatPOENet
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используется устройство: {DEVICE}")

Используется устройство: cuda


In [ ]:
# Вспомогательные функции
def calculate_tour_length(tour, dist_matrix):
    """
    Вычисляет полную длину замкнутого маршрута по матрице расстояний.
    Аргументы:
        tour (Tensor): (batch, n_nodes) – незамкнутый маршрут (без возврата в старт)
        dist_matrix (Tensor): (batch, n_nodes, n_nodes)
    Возвращает:
        length (Tensor): (batch,) длина каждого маршрута в тех же единицах, что и матрица
    """
    batch_size, n_nodes = tour.shape
    length = 0.0
    # суммируем все переходы, кроме возврата к старту
    for i in range(n_nodes - 1):
        length += dist_matrix[torch.arange(batch_size), tour[:, i], tour[:, i + 1]]
    # добавляем возврат от последнего города к первому
    length += dist_matrix[torch.arange(batch_size), tour[:, -1], tour[:, 0]]
    return length

def generate_atsp_data(batch_size, n_nodes, asym_ratio=0.3, device=DEVICE):
    """
    Генерирует случайные асимметричные матрицы расстояний.
    - Создаётся базовая матрица
    - С вероятностью asym_ratio каждый элемент делается асимметричным (независимое случайное значение)
    - Остальные элементы усредняются для симметрии
    """
    dist_matrix = torch.rand(batch_size, n_nodes, n_nodes, device=device)
    asym_mask = torch.rand(batch_size, n_nodes, n_nodes, device=device) < asym_ratio
    dist_matrix[asym_mask] = torch.rand(asym_mask.sum(), device=device)
    sym_mask = ~asym_mask
    dist_matrix[sym_mask] = (dist_matrix[sym_mask] + dist_matrix.transpose(1, 2)[sym_mask]) / 2
    return dist_matrix

def plot_training_curve(losses, title, save_path):
    """Строит график значений функции потерь по эпохам."""
    plt.figure(figsize=(8, 5))
    plt.plot(range(1, len(losses) + 1), losses, marker='o', linestyle='-')
    plt.xlabel('Эпоха')
    plt.ylabel('Средний loss')
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"График обучения сохранён: {save_path}")

In [ ]:
# Обучение модели
def train_model(model, optimizer, num_epochs, batch_size, n_nodes, model_name, save_dir):
    """
    Обучает модель с помощью REINFORCE.
    Для каждой эпохи генерируются 100 батчей синтетических матриц.
    Модель должна возвращать (tour, log_probs) при вызове с return_log_probs=True.
    Лосс: -log_prob * reward, где reward = -tour_length.
    Сохраняет чекпоинты каждую эпоху.
    """
    model.train()
    os.makedirs(save_dir, exist_ok=True)

    epoch_losses = []  # средний лосс за каждую эпоху

    for epoch in range(num_epochs):
        total_loss = 0.0
        num_batches = 100
        pbar = tqdm(range(num_batches), desc=f"Epoch {epoch+1}/{num_epochs}")

        for _ in pbar:
            # Генерация данных
            dist_matrix = generate_atsp_data(batch_size, n_nodes, device=DEVICE)

            # Прямой проход – получаем незамкнутый маршрут и log_prob выбранных действий
            # Все модели теперь поддерживают return_log_probs=True
            tour, log_probs = model(dist_matrix, 0, return_log_probs=True)

            # Вычисляем длину маршрута (отрицательная награда)
            with torch.no_grad():
                tour_length = calculate_tour_length(tour, dist_matrix)
                reward = -tour_length

            # REINFORCE loss: -log_prob * reward (среднее по батчу)
            loss = -(log_probs * reward).mean()

            # Шаг оптимизации
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item()
            pbar.set_postfix({"loss": loss.item()})

        avg_loss = total_loss / num_batches
        epoch_losses.append(avg_loss)
        print(f"Epoch {epoch+1}/{num_epochs}, Average Loss: {avg_loss:.4f}")

        # Сохраняем чекпоинт после каждой эпохи
        checkpoint_path = os.path.join(save_dir, f"{model_name}_epoch_{epoch+1}.pt")
        torch.save(model.state_dict(), checkpoint_path)
        print(f"Модель сохранена: {checkpoint_path}")

    # Сохраняем метрики обучения в JSON
    metrics = {
        'model': model_name,
        'n_nodes': n_nodes,
        'epochs': num_epochs,
        'batch_size': batch_size,
        'final_loss': epoch_losses[-1],
        'loss_history': [float(x) for x in epoch_losses]
    }
    metrics_path = os.path.join(save_dir, f"{model_name}_metrics.json")
    with open(metrics_path, 'w', encoding='utf-8') as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)
    print(f"Метрики сохранены: {metrics_path}")

    # График потерь
    plot_path = os.path.join(save_dir, f"{model_name}_loss.png")
    plot_training_curve(epoch_losses, f"Training Loss: {model_name} (n={n_nodes})", plot_path)

    return epoch_losses

In [ ]:
# Архивирование
def zip_directory(dir_path, zip_name=None):
    import shutil
    if zip_name is None:
        zip_name = dir_path
    shutil.make_archive(zip_name, 'zip', dir_path)
    print(f"Архив создан: {zip_name}.zip")
    return f"{zip_name}.zip"

In [ ]:
#  Основной блок
if __name__ == "__main__":
    # Конфигурация: список моделей и размерностей
    MODEL_CONFIGS = [
        (MatNet, "MatNet", [12, 29]),
        (MatPOENet, "MatPOENet", [12, 29]),
        (PointerNetworkATSP, "PointerNetworkATSP", [12, 29]),
        (AttentionModelATSP, "AttentionModelATSP", [12, 29])
    ]

    NUM_EPOCHS = 20
    BATCH_SIZE = 64
    LEARNING_RATE = 1e-4

    all_save_dirs = []

    for model_class, model_type, sizes in MODEL_CONFIGS:
        for n_nodes in sizes:
            print(f"\nОбучение {model_type} для n={n_nodes}")

            # Инстанцирование модели
            if model_type == "MatNet":
                model = MatNet(n_nodes=n_nodes, embedding_dim=128).to(DEVICE)
            elif model_type == "MatPOENet":
                model = MatPOENet(n_nodes=n_nodes, embedding_dim=128).to(DEVICE)
            elif model_type == "PointerNetworkATSP":
                model = PointerNetworkATSP(n_cities=n_nodes, hidden_dim=128).to(DEVICE)
            elif model_type == "AttentionModelATSP":
                model = AttentionModelATSP(n_cities=n_nodes, embed_dim=128).to(DEVICE)
            else:
                raise ValueError(f"Неизвестный тип модели: {model_type}")

            optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

            model_name = f"{model_type}_{n_nodes}"
            save_dir = f"models_{model_type}_{n_nodes}"
            all_save_dirs.append(save_dir)

            train_model(
                model=model,
                optimizer=optimizer,
                num_epochs=NUM_EPOCHS,
                batch_size=BATCH_SIZE,
                n_nodes=n_nodes,
                model_name=model_name,
                save_dir=save_dir
            )

    # Архивирование всех папок
    print("\nАрхивирование результатов")
    zip_files = []
    for d in all_save_dirs:
        zip_path = zip_directory(d)
        zip_files.append(zip_path)

    print("\nОбучение всех моделей завершено!")


Обучение MatNet для n=12


Epoch 1/20: 100%|██████████| 100/100 [00:03<00:00, 31.23it/s, loss=-105]


Epoch 1/20, Average Loss: -104.6338
Модель сохранена: models_MatNet_12/MatNet_12_epoch_1.pt


Epoch 2/20: 100%|██████████| 100/100 [00:02<00:00, 35.45it/s, loss=-101]


Epoch 2/20, Average Loss: -105.2137
Модель сохранена: models_MatNet_12/MatNet_12_epoch_2.pt


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Архивы скачаны.

Обучение всех моделей завершено!


#GNN

In [ ]:
if __name__ == "__main__":
    all_save_dirs = []

    # обучение GNN-модели (TSPGNNLearner)

    # Параметры для GNN (можно изменить при необходимости)
    GNN_LEARNING_RATE = 1e-4
    GNN_BATCH_SIZE = 64
    GNN_EPOCHS = 20

    gnn_sizes = [12, 29]
    for n_nodes in gnn_sizes:
        model = TSPGNNLearner(n_cities=n_nodes, hidden_dim=128, num_layers=3).to(DEVICE)
        optimizer = optim.Adam(model.parameters(), lr=GNN_LEARNING_RATE)
        model_name = f"TSPGNNLearner_{n_nodes}"
        save_dir = f"models_TSPGNNLearner_{n_nodes}"
        all_save_dirs.append(save_dir)   # добавляем в общий список

        train_model(
            model=model,
            optimizer=optimizer,
            num_epochs=GNN_EPOCHS,
            batch_size=GNN_BATCH_SIZE,
            n_nodes=n_nodes,
            model_name=model_name,
            save_dir=save_dir
        )

    # Архивирование всех папок (включая GNN)
    print("\nАрхивирование результатов")
    zip_files = []
    for d in all_save_dirs:
        zip_path = zip_directory(d)
        zip_files.append(zip_path)

    # Автоматическое скачивание в Colab (опционально)
    try:
        from google.colab import files
        for z in zip_files:
            files.download(z)
        print("Архивы скачаны.")
    except ImportError:
        print("Запуск не в Colab, архивы не скачиваются автоматически.")

    print("\nОбучение всех моделей завершено!")

Epoch 1/20: 100%|██████████| 100/100 [00:05<00:00, 18.80it/s, loss=-103]


Epoch 1/20, Average Loss: -105.0364
Модель сохранена: models_TSPGNNLearner_12/TSPGNNLearner_12_epoch_1.pt


Epoch 2/20: 100%|██████████| 100/100 [00:03<00:00, 32.33it/s, loss=-104]


Epoch 2/20, Average Loss: -105.0457
Модель сохранена: models_TSPGNNLearner_12/TSPGNNLearner_12_epoch_2.pt


Epoch 3/20: 100%|██████████| 100/100 [00:02<00:00, 38.28it/s, loss=-104]


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Архивы скачаны.

Обучение всех моделей завершено!
